In [1]:
import os, random
import cv2 as cv
from pathlib import Path

In [9]:
### dict with class labels
class_lables = {
    "Brown bear": 0,
    "Canary": 1,
    "Cheetah": 2,
    "Crocodile": 3,
    "Elephant": 4,
    "Fox": 5,
    "Goat": 6,
    "Goldfish": 7,
    "Kangaroo": 8,
    "Leopard": 9,
    "Mouse": 10,
    "Ostrich": 11,
    "Panda": 12,
    "Pig": 13,
    "Rabbit": 14,
    "Raccoon": 15,
    "Rhinoceros": 16,
    "Sheep": 17,
    "Woodpecker": 18,
    "Zebra": 19,
}

In [55]:
#### Helper function for transforming label file to correct format
def transform_label_file(input_path, output_path, width, height):
    ### creating output variabel
    output = []
    ### reading input file
    with open(input_path, 'r') as f:
        for line in f:
            content = line.strip().split()
            
            ### crating adjusten and check for extra whitespace in animals with space in between className in the label file.
            i = 0
            label = content[0]
            if len(content) == 6:
                i = 1
                label = label + " " + content[1]

            ### transforming the values in the label file
            x_center = round(((float(content[1+i]) + float(content[3+i])) / 2 ) / width, 4)
            y_center = round(((float(content[2+i]) + float(content[4+i])) / 2 ) / height, 4)
            image_width = round((float(content[3+i]) - float(content[1+i])) / width, 4)
            image_height = round((float(content[4+i]) - float(content[2+i])) / height, 4)
            ### creating the output for the new transformed label file
            output.append(str(class_lables.get(label)) + " " + str(x_center) + " " + str(y_center) + " " + str(image_width) + " " + str(image_height))
    
    ### writing the output to the new label file
    with open(output_path, "w") as f:
        for line in output:
            f.write(f"{line}\n")
            

def pre_process_images(input_path, output_path, random_seed):       
    for root, dirs, file in os.walk(input_path):
        for d in dirs:
            if d != "Label":
                ### reading all the images paths in a folder and adding them to an list for processing
                image_paths = []
                d_path = os.path.join(root, d)
                print(d_path)
                for file in os.listdir(d_path):
                    if file.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        image_paths.append(file)

                ### using random shuffel with seed to create different selections of the images for the dataset.
                random.seed(random_seed)
                random.shuffle(image_paths)                
                
                ### creating splits for partition of dataset
                split_test = int(len(image_paths)*0.1) + 1
                split_val = int(len(image_paths)*0.25) + 1
                split_traing = int(len(image_paths)*0.80) + 1

                count = 0
                ### looping through all images paths
                for file in image_paths:
                    path = os.path.join(d_path, file)
                    img = cv.imread(path)

                    ### creating the paths for the prosseceing of the images
                    label_input_path = d_path + "/Label/" + Path(file).stem + ".txt"                 
                    file_path = d + str(count) + Path(path).suffix
                    label_path = d + str(count) + ".txt"

                    ### creating dataset splits by using count and partition splits, creating splits with abount, testing 10%, validation 15%, training 55% per animal class.
                    if count < split_test:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/test/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/test/" + file_path), img)
                    elif count >= split_test and count < split_val:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/val/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/val/" + file_path), img)                    
                    elif count >= split_val and count < split_traing:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/train/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/train/" + file_path), img)
                    elif count >= split_traing:
                        break
                    count += 1


In [56]:
### variabels for input and output path
input_path = './data/pre/'
output_path = './data/post/'

pre_process_images(input_path, output_path, 15)

./data/pre/Brown bear
./data/pre/Canary
./data/pre/Cheetah
./data/pre/Crocodile
./data/pre/Elephant
./data/pre/Fox
./data/pre/Goat
./data/pre/Goldfish
./data/pre/Kangaroo
./data/pre/Leopard
./data/pre/Mouse
./data/pre/Ostrich
./data/pre/Panda
./data/pre/Pig
./data/pre/Rabbit
./data/pre/Raccoon
./data/pre/Rhinoceros
./data/pre/Sheep
./data/pre/Woodpecker
./data/pre/Zebra
